In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/raw/transactions.csv")

print("Dataset shape:", df.shape)
print(df.head())

Dataset shape: (100000, 21)
  transaction_id   amount payment_method merchant_category  transaction_hour  \
0      TX_000001  1802.11            UPI              Food                 6   
1      TX_000002   955.02           CARD       Electronics                23   
2      TX_000003  2095.80            UPI            Travel                11   
3      TX_000004  5029.27           CARD          Services                 6   
4      TX_000005   867.70            UPI       Electronics                22   

   is_weekend  customer_age_days  customer_avg_amount  amount_deviation  \
0           1               1468               566.67              3.17   
1           1                737               301.70              3.16   
2           0                363               597.73              3.50   
3           0                709              1213.41              4.14   
4           0                603              1259.85              0.69   

   transactions_last_10min  ...  transac

In [2]:
# Remove transaction ID because it has no predictive value
df = df.drop(columns=["transaction_id"])

X = df.drop(columns=["fraud"])
y = df["fraud"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nTarget distribution:")
print(y.value_counts())

Feature shape: (100000, 19)
Target shape: (100000,)

Target distribution:
fraud
0    95585
1     4415
Name: count, dtype: int64


In [3]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical features:")
print(categorical_features)

print("\nNumerical features:")
print(numerical_features)

Categorical features:
['payment_method', 'merchant_category']

Numerical features:
['amount', 'transaction_hour', 'is_weekend', 'customer_age_days', 'customer_avg_amount', 'amount_deviation', 'transactions_last_10min', 'transactions_last_1h', 'transactions_last_24h', 'device_age_days', 'device_transaction_count', 'location_distance_km', 'ip_risk_score', 'failed_attempts_24h', 'previous_fraud_count', 'chargeback_history', 'account_velocity_score']


In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

print("\nTraining fraud distribution:")
print(y_train.value_counts())

print("\nTesting fraud distribution:")
print(y_test.value_counts())

Training samples: 80000
Testing samples: 20000

Training fraud distribution:
fraud
0    76468
1     3532
Name: count, dtype: int64

Testing fraud distribution:
fraud
0    19117
1      883
Name: count, dtype: int64


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

categorical_features = [
    "payment_method",
    "merchant_category"
]

numerical_features = [
    col for col in X.columns
    if col not in categorical_features
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

print("Preprocessor created successfully!")
print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))

Preprocessor created successfully!
Numerical features: 17
Categorical features: 2


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=42
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

print("Logistic Regression trained successfully!")

Logistic Regression trained successfully!


In [7]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)

# Predictions
y_pred = logistic_model.predict(X_test)

# Probability of fraud
y_prob = logistic_model.predict_proba(X_test)[:, 1]

# Metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print("MODEL PERFORMANCE")
print("=" * 40)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}")

print("\nClassification Report")
print("=" * 40)
print(classification_report(y_test, y_pred))

MODEL PERFORMANCE
Accuracy : 0.9894
Precision: 0.8109
Recall   : 0.9909
F1 Score : 0.8919
ROC-AUC  : 0.9994
PR-AUC   : 0.9832

Classification Report
              precision    recall  f1-score   support

           0       1.00      0.99      0.99     19117
           1       0.81      0.99      0.89       883

    accuracy                           0.99     20000
   macro avg       0.91      0.99      0.94     20000
weighted avg       0.99      0.99      0.99     20000



In [8]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline

rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=200,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(X_train, y_train)

print("Random Forest trained successfully!")

Random Forest trained successfully!


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    classification_report
)

rf_pred = rf_model.predict(X_test)
rf_prob = rf_model.predict_proba(X_test)[:, 1]

print("RANDOM FOREST PERFORMANCE")
print("=" * 40)

print("Accuracy :", round(
    accuracy_score(y_test, rf_pred), 4
))

print("Precision:", round(
    precision_score(y_test, rf_pred), 4
))

print("Recall   :", round(
    recall_score(y_test, rf_pred), 4
))

print("F1 Score :", round(
    f1_score(y_test, rf_pred), 4
))

print("ROC-AUC  :", round(
    roc_auc_score(y_test, rf_prob), 4
))

print("PR-AUC   :", round(
    average_precision_score(y_test, rf_prob), 4
))

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        rf_pred
    )
)

RANDOM FOREST PERFORMANCE
Accuracy : 0.9984
Precision: 0.9797
Recall   : 0.9853
F1 Score : 0.9825
ROC-AUC  : 0.9999
PR-AUC   : 0.9986

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     19117
           1       0.98      0.99      0.98       883

    accuracy                           1.00     20000
   macro avg       0.99      0.99      0.99     20000
weighted avg       1.00      1.00      1.00     20000



In [10]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    rf_model,
    "../models/risk_model.pkl"
)

print("Model saved successfully!")
print("Location: models/risk_model.pkl")

Model saved successfully!
Location: models/risk_model.pkl
